# Leitura Incremental de Arquivos

### Objetivo: analisar como funciona o .tell() e o .seek() para realizar a leitura incremental de arquivos, ou seja, conseguir ler apenas as linhas novas sem precisar ler todo o arquivo e sem ter perda de informação.

In [6]:
###### LEITURA INCREMENTAL DE ARQUIVOS - processamento apenas das informações novas ######

#---------------------BIBLIOTECAS UTILIZADAS---------------------#
import time
from datetime import datetime
from file_read_backwards import FileReadBackwards

#-----------------------FUNÇÕES UTILIZADAS-----------------------#

#Para pegar uma informação em alguma linha (linha, posição da informação)
def dado_especifico(linha_desejada, n):
    lista_dados = linha_desejada.split(sep=",")
    lista_dados[0] = lista_dados[0].replace('"', "")
    informacao_desejada = lista_dados[n-1]
    return informacao_desejada

#Para substituir uma linha
def substituir_linha(nome_arquivo, numero_linha, novo_texto):
    # 1. Lê todas as linhas do arquivo
    with open(nome_arquivo, "r", encoding="utf-8") as arquivo:
        linhas = arquivo.readlines()

    # 2. Altera a linha desejada
    linhas[numero_linha] = novo_texto

    # 3. Reescreve o arquivo com a modificação
    with open(nome_arquivo, "w", encoding="utf-8") as arquivo:
        arquivo.writelines(linhas)

#-----------QUANTIDADE DE VEZES QUE O PROGRAMA IRÁ RODAR---------#

while True:
    try:
        quantidade_verificacoes = int(input("Digite a quantidade de vezes que o programa verificará se possui linhas novas: "))
        break
    except:
        continue

while True:
    try:
        tempo_cada_verificacao = int(input("Digite o tempo de pausa entre cada verificação (s): "))
        break
    except:
        continue

#----------------------------PRINCIPAL---------------------------#

#entrada de variáveis
records_total_lista = list()
nome_arquivo_dados_simulados = "/Users/kauad/OneDrive/Documentos/Universidade Federal de Minas Gerais/CEMTEC/Simulação da Estação Solarimétrica/dados_simulados.txt"

record_anterior = 0
with FileReadBackwards("posicao_atual.txt", encoding="utf-8") as frb:
    for linha_record in frb:
        record_anterior = int(linha_record[26:].strip())
        break

for i in range(quantidade_verificacoes):

    #----------------VERIFICAÇÃO DA ÚLTIMA POSIÇÃO---------------#

    file_posicao = open("posicao_atual.txt", "r", encoding="utf-8")
    linha_posicao = file_posicao.readline() #leitura da linha na qual informna a posição do arquivo
    ultima_posicao = linha_posicao[25:].strip() #.strip() remove os espaços em brancos
    ultima_posicao = int(ultima_posicao)
    file_posicao.close()

    #---------MANDA O PROGRAMA IR DIRETO PARA ESSA POSIÇÃO--------#
    file = open(nome_arquivo_dados_simulados, "r", encoding="UTF-8")
    file.seek(ultima_posicao)

    #---------------------ENTRADA DE VARIÁVEIS---------------------#
    posicao_para_gravar = 0
    records_lista = list()

    #------------------LEITURA DE CADA LINHA NOVA------------------#
    while True:
        linha = file.readline()

        #------------------------VERIFICAÇÕES----------------------#

        #Verificação se é o cabeçalho
        linha_lista = linha.split(',')
        lista_verificador_cabecalho = ["TOA5", "TIMESTAMP", "TS"]
        veririficador_cabecalho = linha_lista[0].replace('"',"")
        if veririficador_cabecalho in lista_verificador_cabecalho:
            continue

        #Verificação se a primeira linha é um \n
        if linha == '\n':
            continue

        #Verificação se chegou no final do arquivo
        if linha == "":
            posicao_para_gravar = file.tell()
            file.close()
            break

        #-----------------PROCESSAMENTO DA LINHA NOVA---------------#
        record_linha_nova = int(dado_especifico(linha, 2))
        records_lista.append(record_linha_nova)

        record_anterior += 1

        if len(records_total_lista)!=0:
            substituir_linha("posicao_atual.txt", 1, f"record atual da linha: {records_total_lista[-1]}")

    #------------NÚMERO DE REGISTROS TOTAIS PROCESSADOS-------------#
    records_total_lista = records_total_lista + records_lista

    #--------------GRAVANDO A POSIÇÃO E O RECORD ATUAL--------------#
    substituir_linha("posicao_atual.txt", 0, f"posição atual do arquivo: {posicao_para_gravar}"+"\n")

    #if len(records_total_lista)!=0:
        #substituir_linha("posicao_atual.txt", 1, f"record atual da linha: {records_total_lista[-1]}")

    #---------------------FIM DE UMA VERIFICAÇÃO--------------------#
    print("-----------"*5)
    print(f"VERIFICAÇÃO DE NÚMERO {i+1} CONCLUIDA COM SUCESSO")
    print(f"Gravações processadas: {records_lista}")
    print(f"Quantidade de linhas novas: {len(records_lista)}")
    print("-----------"*5)
    print("\n")

    time.sleep(tempo_cada_verificacao)

#---------------------FIM DO PROGRAMA--------------------#
print("PROGRAMA FINALIZADO")
print(f"TOTAL DE GRAVAÇÕES PROCESSADAS: {records_total_lista}")
print(f"QUANTIDADE TOTAL DE LINH AS NOVAS: {len(records_total_lista)}")

-------------------------------------------------------
VERIFICAÇÃO DE NÚMERO 1 CONCLUIDA COM SUCESSO
Gravações processadas: [17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27]
Quantidade de linhas novas: 11
-------------------------------------------------------


PROGRAMA FINALIZADO
TOTAL DE GRAVAÇÕES PROCESSADAS: [17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27]
QUANTIDADE TOTAL DE LINH AS NOVAS: 11
